# Private Fine-tuning with Lip-DP-Diffusion

# Lip-DP-Diffusion: Differentially Private Training for Diffusion Models

This notebook implements **Lip-DP-Diffusion**, a novel mechanism for training 
Denoising Diffusion Probabilistic Models (DDPMs) under differential privacy 
guarantees. Rather than relying on DP-SGD's per-sample gradient clipping, 
Lip-DP-Diffusion enforces Lipschitz constraints directly on the model weights, 
enabling data-independent, per-layer sensitivity computation and calibrated 
noise injection.

---

## What This Notebook Does

- Trains a class-conditional U-Net diffusion model on MNIST and Fashion-MNIST 
  under differential privacy
- Computes per-layer gradient sensitivities analytically via two traversals 
  of the U-Net DAG: a forward pass for input bound propagation and a backward 
  pass for gradient Lipschitz tracking
- Injects calibrated Gaussian noise per layer based on each layer's structural 
  position in the network, rather than a single global clipping norm
- Enforces spectral norm constraints on all trainable layers via power 
  iteration clipping, with inequality constraints: layers with $\sigma(W) < C$ 
  are left untouched and receive proportionally less noise
- Tracks the privacy budget using the Rényi Differential Privacy (RDP) 
  accountant with Poisson subsampling amplification
- Supports **transfer learning**: public pre-training on Fashion-MNIST 
  without any privacy noise (spectral constraints enforced but $\sigma=0$), 
  followed by private fine-tuning on MNIST with Lip-DP-Diffusion*
  ($\sigma > 0$, full DP guarantee). Pre-training within the constrained weight 
  space is critical: weights must already satisfy the spectral norm bounds before 
  fine-tuning begins, so that the initial `ClipWeights` step does not destroy 
  learned features.

---

## Privacy Guarantee

The formal $(\varepsilon, \delta)$-DP guarantee is computed at the end of 
training by converting the accumulated RDP bound. The accountant receives only 
the noise  $\sigma$ and sampling rate $q = B/N$ at each step(both 
data-independent) and requires no knowledge of the per-layer sensitivity 
structure.

---

## Evaluation

We evaluate the Phase 2 model at two privacy budget milestones (ε ≈ 5 and ε ≈ 8)
and with two guidance settings (w=0.0 and w=2.0) for both EMA and live weights.

### Metrics

| Metric | What it measures |
|--------|------------------|
| **FID** | Fréchet distance between real and generated Inception-V3 features (↓ = better) |
| **IS**  | Inception Score — sharpness + diversity of generated images (↑ = better) |
| **TSTR accuracy** | Train-on-Synthetic / Test-on-Real: classifiers trained on 60k synthetic images evaluated on 10k real MNIST digits |

### Evaluation variants

- **EMA vs Live weights**: EMA weights are used for generation (smoother, less noisy)
- **w=0.0**: Unconditional sampling (no classifier-free guidance)
- **w=2.0**: Classifier-free guidance (sharper digits, more class-conditional)

In [1]:
!pip install -q opacus torch-fidelity


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
print(torch.cuda.device_count())      
print(torch.cuda.current_device())    
print(torch.cuda.get_device_name(0))  

1
0
NVIDIA A100-SXM4-40GB


In [3]:
import os, math, json, random
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import MNIST, FashionMNIST
from torchvision.utils import save_image, make_grid
from torch.utils.data import DataLoader

from opacus.accountants import RDPAccountant
from opacus.utils.uniform_sampler import UniformWithReplacementSampler
from torch_fidelity import calculate_metrics
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.backends.cudnn.benchmark = True

device: cuda


In [4]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE_DIR = "/content/drive/MyDrive/diffusion_experiments"
except Exception:
    BASE_DRIVE_DIR = "./diffusion_experiments"

DATE_STR = datetime.now().strftime("%d.%m.%Y")
RUN_DIR = os.path.join(BASE_DRIVE_DIR, DATE_STR)
os.makedirs(RUN_DIR, exist_ok=True)
print("RUN_DIR:", RUN_DIR)

def make_exp_dir(exp_name: str):
    exp_dir = os.path.join(RUN_DIR, exp_name)
    os.makedirs(exp_dir, exist_ok=True)
    for sub in ["checkpoints", "samples", "metrics", "fid_is", "synthetic_data", "diagnostics"]:
        os.makedirs(os.path.join(exp_dir, sub), exist_ok=True)
    return exp_dir

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

RUN_DIR: ./diffusion_experiments/14.02.2026


### Data

In [5]:
# same transform will be used for test/train
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 2) - 1),
])
# test_tfm = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Lambda(lambda x: (x * 2) - 1),
# ])

In [6]:
# 1. Public Data: FashionMNIST
fmnist_train = FashionMNIST(root="./data", train=True, transform=tfm, download=True)
fmnist_test  = FashionMNIST(root="./data", train=False, transform=tfm, download=True)

# 2. Private Data: MNIST
mnist_train = MNIST(root="./data", train=True, transform=tfm, download=True)
mnist_test  = MNIST(root="./data", train=False, transform=tfm, download=True)

In [7]:
max_norm = 0.0
for x, _ in mnist_train:
    # Flatten to (1, -1) to get vector norm per image
    norm = torch.linalg.norm(x.view(-1))
    if norm > max_norm:
        max_norm = norm
print(f"Computed max dataset norm: {max_norm}") 
X0_PUBLIC = float(max_norm)

Computed max dataset norm: 28.0


In [8]:
# Before creating DPBoundedEngine
max_norm = 0.0
for x, _ in fmnist_train:
    # Flatten to (1, -1) to get vector norm per image
    norm = torch.linalg.norm(x.view(-1))
    if norm > max_norm:
        max_norm = norm
print(f"Computed max dataset norm: {max_norm}") # Likely ~10-12
X0_PUBLIC = float(max_norm)

Computed max dataset norm: 27.663667678833008


### Model Architecture: Conditional U-Net

In [ ]:
class LipSiLU(nn.Module):
    # conservative scaling: make SiLU ~ 1-Lipschitz
    def __init__(self, scale: float = 1/1.0998393):
        super().__init__()
        self.scale = float(scale)
    def forward(self, x):
        return self.scale * F.silu(x)

In [ ]:
class LipschitzGroupNorm(nn.Module):
    """
    Conservative GroupNorm: denominator clamped >= 1 to be non-expansive.
    """
    def __init__(self, num_groups, num_channels):
        super().__init__()
        self.num_groups = num_groups
        self.num_channels = num_channels

    def forward(self, x):
        B, C, H, W = x.shape
        G = self.num_groups
        x = x.view(B, G, C // G, H, W)
        mean = x.mean(dim=[2, 3, 4], keepdim=True)
        var  = x.var(dim=[2, 3, 4], keepdim=True, unbiased=False)
        std  = (var + 1e-5).sqrt()
        denom = torch.clamp(std, min=1.0)
        x_norm = (x - mean) / denom
        return x_norm.view(B, C, H, W)

In [ ]:
def get_time_embedding(time_steps, t_emb_dim, device):
    half_dim = t_emb_dim // 2
    emb = math.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
    emb = time_steps[:, None].float() * emb[None, :]
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if t_emb_dim % 2 == 1:
        emb = F.pad(emb, (0, 1, 0, 0))
    return emb

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
    def forward(self, x):
        return self.conv(self.up(x))

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, use_lip_norm=True):
        super().__init__()
        self.act = LipSiLU() #if use_lip_norm else nn.SiLU()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        self.time_mlp = nn.Sequential(
            self.act,
            nn.Linear(t_emb_dim, out_channels * 2),
        )

        self.skip_conv = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

        if use_lip_norm:
            self.gn1 = LipschitzGroupNorm(num_groups=8, num_channels=out_channels)
            self.gn2 = LipschitzGroupNorm(num_groups=8, num_channels=out_channels)
        else:
            self.gn1 = nn.GroupNorm(8, out_channels, affine=False)
            self.gn2 = nn.GroupNorm(8, out_channels, affine=False)

    def forward(self, x, emb):
        h = self.conv1(x)
        h = self.gn1(h)

        time_emb = self.time_mlp(emb).unsqueeze(-1).unsqueeze(-1)
        scale, shift = time_emb.chunk(2, dim=1)
        scale = torch.tanh(scale)
        h = self.act(h * (1 + scale) + shift)

        h = self.conv2(h)
        h = self.gn2(h)
        h = self.act(h)

        skip = self.skip_conv(x)
        return (h + skip)/ math.sqrt(2.0)

In [9]:
class CondDiffUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, t_emb_dim=128, n_classes=11, use_lip_norm=True):
        super().__init__()
        self.t_emb_dim = t_emb_dim
        self.n_classes = n_classes

        self.label_emb = nn.Embedding(n_classes, t_emb_dim)
        act = LipSiLU() # if use_lip_norm else nn.SiLU()

        self.time_proj = nn.Sequential(
            nn.Linear(t_emb_dim, t_emb_dim),
            act,
            nn.Linear(t_emb_dim, t_emb_dim),
        )

        # Down
        self.down1 = ResBlock(in_channels, base_channels, t_emb_dim, use_lip_norm)
        self.pool1 = nn.Conv2d(base_channels, base_channels, 4, stride=2, padding=1)

        self.down2 = ResBlock(base_channels, base_channels*2, t_emb_dim, use_lip_norm)
        self.pool2 = nn.Conv2d(base_channels*2, base_channels*2, 4, stride=2, padding=1)

        self.down3 = ResBlock(base_channels*2, base_channels*4, t_emb_dim, use_lip_norm)

        # Bottleneck
        self.bot1 = ResBlock(base_channels*4, base_channels*4, t_emb_dim, use_lip_norm)

        # Up
        self.up1     = UpBlock(base_channels*4, base_channels*2)
        self.up_res1 = ResBlock(base_channels*4, base_channels*2, t_emb_dim, use_lip_norm)

        self.up2     = UpBlock(base_channels*2, base_channels)
        self.up_res2 = ResBlock(base_channels*2, base_channels, t_emb_dim, use_lip_norm)

        self.out_conv = nn.Conv2d(base_channels, in_channels, 3, padding=1)

    def forward(self, x, t, y):
        t_emb = get_time_embedding(t, self.t_emb_dim, x.device)
        y_emb = self.label_emb(y)
        emb = self.time_proj(t_emb + y_emb)

        x1 = self.down1(x, emb)
        x2 = self.pool1(x1)

        x2 = self.down2(x2, emb)
        x3 = self.pool2(x2)

        x3 = self.down3(x3, emb)
        bn = self.bot1(x3, emb)

        u1 = self.up1(bn)
        u1 = torch.cat([u1, x2], dim=1) #/ math.sqrt(2.0)
        u1 = self.up_res1(u1, emb)

        u2 = self.up2(u1)
        u2 = torch.cat([u2, x1], dim=1) #/ math.sqrt(2.0)
        u2 = self.up_res2(u2, emb)

        return self.out_conv(u2)

### DDPM schedule + v-prediction

In [10]:
class DDPMConfig:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.T = timesteps
        self.beta_start = beta_start
        self.beta_end = beta_end

#linear schedule
def make_ddpm_schedule(cfg: DDPMConfig, device):
    betas = torch.linspace(cfg.beta_start, cfg.beta_end, cfg.T, device=device)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars

In [11]:
@torch.no_grad()
def v_to_eps_and_x0(x_t, t, v, alpha_bars):
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    sqrt_ab   = torch.sqrt(ab)
    sqrt_1mab = torch.sqrt(1.0 - ab)
    x0  = sqrt_ab * x_t - sqrt_1mab * v
    eps = sqrt_1mab * x_t + sqrt_ab * v
    return eps, x0

def compute_v_target(x0, eps, t, alpha_bars):
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    return torch.sqrt(ab) * eps - torch.sqrt(1.0 - ab) * x0

In [12]:
class EMA:
    def __init__(self, model: nn.Module, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.detach().clone()

    @torch.no_grad()
    def update(self, model):
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            self.shadow[name].mul_(self.decay).add_(p.detach(), alpha=(1.0 - self.decay))

    @torch.no_grad()
    def copy_to(self, model):
        for name, p in model.named_parameters():
            if name in self.shadow:
                p.data.copy_(self.shadow[name].data)

In [13]:
def l2_clip_per_sample(x, clip):
    B = x.shape[0]
    flat = x.view(B, -1)
    norms = torch.linalg.norm(flat, dim=1).clamp(min=1e-12)
    scale = (clip / norms).clamp(max=1.0)
    return (flat * scale[:, None]).view_as(x)

In [ ]:
# loss function - bounded MSE loss
def clipped_residual_mse(v_pred, v_target, R):
    """
    Bounds ||∂loss/∂v_pred|| by clipping residual per-sample to radius R.
    """
    r = v_pred - v_target
    r_clipped = l2_clip_per_sample(r, R)
    return r_clipped.pow(2).mean()

In [ ]:
# loss function - Huber loss
def huber_loss(v_pred, v_target, delta, reduction="mean"):
    """
    SmoothL1 / Huber loss.

    PyTorch SmoothL1Loss with beta=delta is Huber:
      - quadratic when |r| < delta
      - linear when |r| >= delta

    Bounds the per-element gradient dℓ/dr to <= delta (up to reduction scaling).
    """
    return F.smooth_l1_loss(v_pred, v_target, beta=float(delta), reduction=reduction)

In [14]:
# weight spectral-norm clipping + conservative conv factors
@torch.no_grad()
def power_iter_weight_matrix(w, iters=2, eps=1e-12):
    # w: (out, in_flat)
    u = torch.randn(w.size(0), device=w.device)
    u = u / (u.norm() + eps)
    for _ in range(iters):
        v = (w.t() @ u); v = v / (v.norm() + eps)
        u = (w @ v);     u = u / (u.norm() + eps)
    sigma = torch.dot(u, w @ v).abs()
    return sigma

@torch.no_grad()
def clip_linear_like(module, C, iters=2):
    """
    Clips Conv2d/Linear by spectral norm of flattened weights.
    For conv this is NOT the true conv operator norm; we compensate conservatively
    in sensitivity with a kernel-size factor.
    """
    if not hasattr(module, "weight") or module.weight is None:
        return 1.0
    w = module.weight
    w_mat = w.view(w.shape[0], -1)
    sigma = power_iter_weight_matrix(w_mat, iters=iters)
    if sigma > C:
        module.weight.mul_(C / (sigma + 1e-12))
        sigma = torch.tensor(C, device=w.device)
    return float(sigma)

In [ ]:
def conv_param_factor(conv: nn.Conv2d):
    kh, kw = conv.kernel_size
    # conservative factor (paper-style bound for ∂f/∂W scales with sqrt(window_size))
    return math.sqrt(kh * kw)

In [15]:
def resblock_lipschitz(block_name, model, sigma_dict):
    """
    block output: (h + skip)/sqrt(2)
    h path ~ sigma(conv2)*sigma(conv1) (GN/act/time assumed 1-Lip)
    skip path ~ 1 if identity else sigma(skip_conv)
    """
    inv_sqrt2 = 1.0 / math.sqrt(2.0)
    s1 = sigma_dict.get(f"{block_name}.conv1", 1.0)
    s2 = sigma_dict.get(f"{block_name}.conv2", 1.0)
    blk = getattr(model, block_name)
    if isinstance(blk.skip_conv, nn.Conv2d):
        s_skip = sigma_dict.get(f"{block_name}.skip_conv", 1.0)
    else:
        s_skip = 1.0
    return inv_sqrt2 * (s2 * s1 + s_skip)


def upblock_lipschitz(block_name, sigma_dict):
    return sigma_dict.get(f"{block_name}.conv", 1.0)

def concat_scaled_norm_bound(a_bound, b_bound):
    # concat then /sqrt(2): ||[a,b]||/sqrt(2) <= max(||a||,||b||)
    return max(a_bound, b_bound)

In [ ]:
@torch.no_grad()
def compute_public_activation_bounds(model, sigma, X0):
    """
    Deterministic public propagation of activation norm upper bounds
    through this specific U-Net DAG.
    """
    X = {}
    X["x0"] = float(X0)

    L_down1 = resblock_lipschitz("down1", model, sigma)
    X["x1"] = L_down1 * X["x0"]

    X["x2_in"] = sigma.get("pool1", 1.0) * X["x1"]

    L_down2 = resblock_lipschitz("down2", model, sigma)
    X["x2"] = L_down2 * X["x2_in"]

    X["x3_in"] = sigma.get("pool2", 1.0) * X["x2"]

    L_down3 = resblock_lipschitz("down3", model, sigma)
    X["x3"] = L_down3 * X["x3_in"]

    L_bot = resblock_lipschitz("bot1", model, sigma)
    X["bn"] = L_bot * X["x3"]

    L_up1 = upblock_lipschitz("up1", sigma)
    X["u1_up"] = L_up1 * X["bn"]
    X["u1_cat"] = concat_scaled_norm_bound(X["u1_up"], X["x2"])
    L_upres1 = resblock_lipschitz("up_res1", model, sigma)
    X["u1"] = L_upres1 * X["u1_cat"]

    L_up2 = upblock_lipschitz("up2", sigma)
    X["u2_up"] = L_up2 * X["u1"]
    X["u2_cat"] = concat_scaled_norm_bound(X["u2_up"], X["x1"])
    L_upres2 = resblock_lipschitz("up_res2", model, sigma)
    X["u2"] = L_upres2 * X["u2_cat"]

    return X

### DP Engine

In [16]:
class DPBoundedEngine:
    def __init__(
        self,
        model,
        optimizer,
        C_weight,
        noise_multiplier,
        X0_public,
        Huber_R,
        pi_iters=2,
        clip_every=10,
    ):
        self.model = model
        self.opt = optimizer
        self.C = float(C_weight)
        self.noise_multiplier = float(noise_multiplier)

        self.X0 = float(X0_public)
        self.R = float(Huber_R)

        self.pi_iters = int(pi_iters)
        self.clip_every = int(clip_every)

        self.step_count = 0
        self.sigma = {}
        self.Xbounds = {}
        self.delta_param = {}
        self.delta_global = 0.0
        self.P = 1*28*28

        self.refresh_bounds(force=True)
        
    @torch.no_grad()
    def add_noise_and_step(self, B: int, debug: bool = True):
        """
        One DP update step with PER-LAYER noise injection:
          - refresh public bounds periodically
          - compute layer-wise sensitivities (delta_k)
          - add Gaussian noise N(0, (noise_multiplier * delta_k)^2 I) to specific layer grads
          - optimizer step and zero grads

        Args:
            B: effective batch size for this step (from Poisson sampler)
            debug: if True, prints signal-vs-noise diagnostics occasionally
        """
        self.step_count += 1
        self.refresh_bounds(force=False)

        # 1) Compute sensitivities
        # This populates self.delta_param (per layer) and returns delta_global
        delta_global = float(self.compute_global_sensitivity(B))

        # Track which params have received noise to ensure we don't miss any (e.g. Embeddings)
        noised_params = set()

        if self.noise_multiplier > 0: # when noise_multiplier = 0 basic Lipschitz constrained architecture with no noise training
            # 2) Iterate modules to add specific per-layer noise
            for name, mod in self.model.named_modules():
                if isinstance(mod, (nn.Conv2d, nn.Linear)):
                    if hasattr(mod, "weight") and mod.weight is not None and mod.weight.grad is not None:
                        # get specific sensitivity for this layer, fallback to global if missing (shouldn't happen if names match)
                        delta_k = self.delta_param.get(name, delta_global)

                        # calc noise std specific to this layer  N(0, sigma^2 * delta_k^2 * I)
                        layer_noise_std = self.noise_multiplier * delta_k

                        # add noise to weights
                        mod.weight.grad.add_(torch.randn_like(mod.weight.grad) * layer_noise_std)
                        noised_params.add(mod.weight)

                        if mod.bias is not None and mod.bias.grad is not None:
                            mod.bias.grad.add_(torch.randn_like(mod.bias.grad) * layer_noise_std)
                            noised_params.add(mod.bias)

            # conservative Global Sensitivity noise to ensure DP safety no for any other param
            fallback_std = self.noise_multiplier * delta_global
            for p in self.model.parameters():
                if p.grad is not None and p not in noised_params:
                    p.grad.add_(torch.randn_like(p.grad) * fallback_std)

        # diagnostics
        if debug and (self.step_count % 50 == 0 or self.step_count <= 5):
            grad_rms_list = []
            for p in self.model.parameters():
                if p.grad is None: continue
                grad_rms_list.append(p.grad.detach().pow(2).mean().sqrt().item())

            avg_grad_rms = float(np.mean(grad_rms_list)) if len(grad_rms_list) > 0 else 0.0
            # Use global delta for rough SNR reporting
            global_noise_std = self.noise_multiplier * delta_global
            snr = (avg_grad_rms / (global_noise_std + 1e-12)) if global_noise_std > 0 else float("inf")

            print(
                f"[DP step {self.step_count}] B={int(B)} | Δ_global={delta_global:.3e} "
                f"| avg_grad_RMS={avg_grad_rms:.3e} | Approx SNR={snr:.3e}"
            )

        # 5) Optimizer step
        self.opt.step()
        self.opt.zero_grad(set_to_none=True)


    @torch.no_grad()
    def refresh_bounds(self, force: bool = False):
        if (not force) and (self.step_count % self.clip_every != 0):
            return

        self.sigma = {}
        for name, mod in self.model.named_modules():
            if isinstance(mod, (nn.Conv2d, nn.Linear)):
                sig = clip_linear_like(mod, self.C, iters=self.pi_iters)
                self.sigma[name] = min(self.C, float(sig))

        def get_sig(name):
          return self.sigma.get(name, self.C)

        self.Xbounds = compute_public_activation_bounds(self.model, self.sigma, X0=self.X0)
        # time emb inputs
        t_bound = math.sqrt(self.model.t_emb_dim)

        sig_tp0 = get_sig("time_proj.0")
        sig_tp2 = get_sig("time_proj.2")
        emb_bound = sig_tp2 * sig_tp0 * t_bound
        
        shift_bounds = {
            "down1": get_sig("down1.time_mlp.1") * emb_bound,
            "down2": get_sig("down2.time_mlp.1") * emb_bound,
            "down3": get_sig("down3.time_mlp.1")  * emb_bound,
            "bot1":  get_sig("bot1.time_mlp.1") * emb_bound,
            "up_res1":get_sig("up_res1.time_mlp.1") * emb_bound,
            "up_res2":get_sig("up_res2.time_mlp.1") * emb_bound }
        

        self.Xin = {
            # down1
            "down1.conv1": self.Xbounds["x0"],
            "down1.conv2":     get_sig("down1.conv1") * self.Xbounds["x0"] + shift_bounds["down1"],
            # "down1.conv2": self.Xbounds["x0"],
            "down1.skip_conv": self.Xbounds["x0"],

            # pool1
            "pool1":       self.Xbounds["x1"],

            # down2
            "down2.conv1": self.Xbounds["x2_in"],
            # "down2.conv2": self.Xbounds["x2_in"],
            "down2.conv2":     get_sig("down2.conv1") * self.Xbounds["x2_in"]+ shift_bounds["down2"],
            "down2.skip_conv": self.Xbounds["x2_in"],

            # pool2
            "pool2":       self.Xbounds["x2"],


            # down3
            "down3.conv1": self.Xbounds["x3_in"],
            # "down3.conv2": self.Xbounds["x3_in"],
            "down3.conv2":     get_sig("down3.conv1") * self.Xbounds["x3_in"]+shift_bounds["down3"],
            "down3.skip_conv": self.Xbounds["x3_in"],

            # bot1
            "bot1.conv1":  self.Xbounds["x3"],
            # "bot1.conv2":  self.Xbounds["x3"],
            "bot1.conv2":     get_sig("bot1.conv1") * self.Xbounds["x3"]+ shift_bounds["bot1"],
            "bot1.skip_conv": self.Xbounds["x3"],

            #up1
            "up1.conv":      self.Xbounds["bn"],

            #up_res1
            "up_res1.conv1": self.Xbounds["u1_cat"],
            # "up_res1.conv2": self.Xbounds["u1_cat"],
            "up_res1.conv2":     get_sig("up_res1.conv1") * self.Xbounds["u1_cat"] + shift_bounds["up_res1"],
            "up_res1.skip_conv": self.Xbounds["u1_cat"],

            #up2
            "up2.conv":      self.Xbounds["u1"],

            #up_res2
            "up_res2.conv1": self.Xbounds["u2_cat"],
            # "up_res2.conv2": self.Xbounds["u2_cat"],
            "up_res2.conv2":     get_sig("up_res2.conv1") * self.Xbounds["u2_cat"] + shift_bounds["up_res2"],
            "up_res2.skip_conv": self.Xbounds["u2_cat"],

            #output
            "out_conv":    self.Xbounds["u2"],}


        self.Xin["time_proj.0"] = t_bound
        self.Xin["time_proj.2"] = t_bound


        for blk in ["down1", "down2", "down3", "bot1", "up_res1", "up_res2"]:
            self.Xin[f"{blk}.time_mlp.1"] = emb_bound

        L_post = {}
        inv_sqrt2 = 1.0 / math.sqrt(2.0)
        L_emb_total = 0.0

        # helper for time MLP injection
        def accumulate_time_sens(blk_name, L_at_injection_point):
            L_post[f"{blk_name}.time_mlp.1"] = L_at_injection_point
            sig_linear = get_sig(f"{blk_name}.time_mlp.1")
            return L_at_injection_point * sig_linear

        # helper for ResBlock Skip conn
        # block output y = (Main(x) + Skip(x)) / sqrt(2)
        # grad flow to Skip: L_skip = L_block_output * (1/sqrt(2))
        def handle_skip(blk_name, L_block_out):
            skip_name = f"{blk_name}.skip_conv"
            if skip_name in self.sigma: # Only if trainable Conv2d
                L_post[skip_name] = L_block_out * inv_sqrt2

        #  A.output 
        L_post["out_conv"] = 1.0
        L_u2 = 1.0 * get_sig("out_conv")

        #  B. up_res2
        handle_skip("up_res2", L_u2)
        L_post["up_res2.conv2"] = L_u2 * inv_sqrt2
        L_post["up_res2.conv1"] = (L_u2 * inv_sqrt2) * get_sig("up_res2.conv2")

        L_conv2_in = (L_u2 * inv_sqrt2)
        L_emb_total += accumulate_time_sens("up_res2", L_conv2_in)

        L_u2_cat = L_u2 * resblock_lipschitz("up_res2", self.model, self.sigma)
        L_u2_up = L_u2_cat * inv_sqrt2
        L_x1_branch = L_u2_cat * inv_sqrt2

        # C. up2 
        L_post["up2.conv"] = L_u2_up
        L_u1 = L_u2_up * upblock_lipschitz("up2", self.sigma)

        # D. up_res1 
        handle_skip("up_res1", L_u1)
        L_post["up_res1.conv2"] = L_u1 * inv_sqrt2
        L_post["up_res1.conv1"] = (L_u1 * inv_sqrt2) * get_sig("up_res1.conv2")

        L_conv2_in = (L_u1 * inv_sqrt2)
        L_emb_total += accumulate_time_sens("up_res1", L_conv2_in)

        L_u1_cat = L_u1 * resblock_lipschitz("up_res1", self.model, self.sigma)
        L_u1_up = L_u1_cat * inv_sqrt2
        L_x2_branch = L_u1_cat * inv_sqrt2

        # E. up1
        L_post["up1.conv"] = L_u1_up
        L_bn = L_u1_up * upblock_lipschitz("up1", self.sigma)

        # F. bot1 
        handle_skip("bot1", L_bn)
        L_post["bot1.conv2"] = L_bn * inv_sqrt2
        L_post["bot1.conv1"] = (L_bn * inv_sqrt2) * get_sig("bot1.conv2")

        L_conv2_in = (L_bn * inv_sqrt2)
        L_emb_total += accumulate_time_sens("bot1", L_conv2_in)

        L_x3 = L_bn * resblock_lipschitz("bot1", self.model, self.sigma)

        # G. down3
        handle_skip("down3", L_x3)
        L_post["down3.conv2"] = L_x3 * inv_sqrt2
        L_post["down3.conv1"] = (L_x3 * inv_sqrt2) * get_sig("down3.conv2")

        L_conv2_in = (L_x3 * inv_sqrt2)
        L_emb_total += accumulate_time_sens("down3", L_conv2_in)

        L_x3_in = L_x3 * resblock_lipschitz("down3", self.model, self.sigma)

        # H. pool2
        L_post["pool2"] = L_x3_in
        L_x2_main = L_x3_in * get_sig("pool2")
        L_x2_total = L_x2_main + L_x2_branch

        # I. down2
        handle_skip("down2", L_x2_total)
        L_post["down2.conv2"] = L_x2_total * inv_sqrt2
        L_post["down2.conv1"] = (L_x2_total * inv_sqrt2) * get_sig("down2.conv2")

        L_conv2_in = (L_x2_total * inv_sqrt2)
        L_emb_total += accumulate_time_sens("down2", L_conv2_in)

        L_x2_in = L_x2_total * resblock_lipschitz("down2", self.model, self.sigma)

        # J. pool1
        L_post["pool1"] = L_x2_in
        L_x1_main = L_x2_in * get_sig("pool1")
        L_x1_total = L_x1_main + L_x1_branch

        # K. down1 
        handle_skip("down1", L_x1_total)
        L_post["down1.conv2"] = L_x1_total * inv_sqrt2
        L_post["down1.conv1"] = (L_x1_total * inv_sqrt2) * get_sig("down1.conv2")

        L_conv2_in = (L_x1_total * inv_sqrt2)
        L_emb_total += accumulate_time_sens("down1", L_conv2_in)

        # L. Time Projection
        L_post["time_proj.2"] = L_emb_total
        L_mid = L_emb_total * get_sig("time_proj.2")
        L_post["time_proj.0"] = L_mid

        self.L_post = L_post

    def compute_global_sensitivity(self, B: int):
        # output dimension per sample (MNIST v-pred: 1*28*28)
        P = self.P

        # Huber(mean) => ||dℓ/dv_i||_2 <= R / (B * sqrt(P))
        G = (self.R / math.sqrt(float(P))) / max(int(B), 1) # R is Huber delta


        self.delta_param = {}
        sq_sum = 0.0

        for name, mod in self.model.named_modules():
            if not isinstance(mod, (nn.Conv2d, nn.Linear)) or mod.weight is None:
                continue

            x_in = float(self.Xin.get(name, self.Xbounds["x0"]))  
            Kp = 1.0 #conv_param_factor(mod) if isinstance(mod, nn.Conv2d) else 1.0

            Lpost = self.L_post[name]
            delta = G * Lpost * Kp * x_in

            self.delta_param[name] = float(delta)
            sq_sum += float(delta) ** 2

        self.delta_global = math.sqrt(sq_sum) if sq_sum > 0 else 0.0

        return self.delta_global

### Trainer (Poisson sampling + K noise draws)

In [17]:
class DiffusionDPTrainer:
    def __init__(
        self,
        exp_name,
        model,
        ddpm_cfg,
        train_dataset,
        test_dataset,
        q_sample_rate,
        steps,
        lr,
        K_noise_mult,
        ema_decay,
        # DP params
        C_weight,
        noise_multiplier,
        delta,
        # public bounds
        X0_public,
        Huber_R,
        # speed knobs
        pi_iters=2,
        clip_every=10,
    ):
        self.exp_name = exp_name
        self.exp_dir = make_exp_dir(exp_name)

        self.cfg = ddpm_cfg
        self.betas, self.alphas, self.alpha_bars = make_ddpm_schedule(ddpm_cfg, device=device)

        self.model = model.to(device)
        self.ema = EMA(self.model, decay=ema_decay)

        self.opt = torch.optim.Adam(self.model.parameters(), lr=lr)

        self.K = int(K_noise_mult)
        self.steps = int(steps)

        self.q = float(q_sample_rate)
        self.delta = float(delta)
        self.noise_multiplier = float(noise_multiplier)
        self.eps10_saved = False
        self.eps5_saved = False 
        self.eps8_saved = False

        self.train_loader = DataLoader(
            train_dataset,
            batch_sampler=UniformWithReplacementSampler(
                num_samples=len(train_dataset),
                sample_rate=self.q,
            ),
            num_workers=0,
            pin_memory=True,
            persistent_workers=False,
        )
        self.test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

        self.accountant = RDPAccountant()

        self.dp_engine = DPBoundedEngine(
            model=self.model,
            optimizer=self.opt,
            C_weight=C_weight,
            noise_multiplier=noise_multiplier,
            X0_public=X0_public,
            Huber_R=Huber_R,
            pi_iters=pi_iters,
            clip_every=clip_every,
        )

        self.history = []

        # define "epoch" under Poisson as ~1/q steps
        self.steps_per_epoch = max(1, int(round(1.0 / self.q)))

    def get_epsilon(self):
        if self.noise_multiplier == 0: 
            return 0.0
        return float(self.accountant.get_epsilon(delta=self.delta))

    def save_checkpoint(self, step):
        ckpt_dir = os.path.join(self.exp_dir, "checkpoints")
        os.makedirs(ckpt_dir, exist_ok=True)
        torch.save(self.model.state_dict(), os.path.join(ckpt_dir, "model_latest.pth"))
        torch.save(self.ema.shadow, os.path.join(ckpt_dir, "ema_latest.pth"))
        torch.save({"step": step, "sigma": self.dp_engine.sigma, "Xbounds": self.dp_engine.Xbounds},
                   os.path.join(ckpt_dir, "bounds_latest.pt"))
        # occasional numbered ckpt
        if step % (self.steps_per_epoch) == 0:
            torch.save(self.model.state_dict(), os.path.join(ckpt_dir, f"model_step{step}.pth"))
            torch.save(self.ema.shadow, os.path.join(ckpt_dir, f"ema_step{step}.pth"))

    @torch.no_grad()
    def sample_grid(self, step, nrow=12, guidance_scale=2.0):
        # 1. SETUP
        self.model.eval()
        # Save a backup of the live weights so we can restore them later
        live_state_dict = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
        
        # Helper function to run the actual loop (avoids code duplication)
        def generate_and_save(prefix, use_guidance):
            n = nrow * nrow
            x = torch.randn(n, 1, 28, 28, device=device)
            # Digits 0-9 repeating
            y = torch.tensor([i % 10 for i in range(n)], device=device, dtype=torch.long)
            y_null = torch.full_like(y, 10) # Assuming 10 is null class

            for i in reversed(range(self.cfg.T)):
                t = torch.full((n,), i, device=device, dtype=torch.long)
                
                if use_guidance > 0.0:
                    v_cond = self.model(x, t, y)
                    v_uncond = self.model(x, t, y_null)
                    v_pred = v_uncond + use_guidance * (v_cond - v_uncond)
                else:
                    v_pred = self.model(x, t, y)

                eps_pred, _ = v_to_eps_and_x0(x, t, v_pred, self.alpha_bars)
                alpha = self.alphas[i]
                ab = self.alpha_bars[i]
                beta = self.betas[i]
                
                mean = (1.0 / torch.sqrt(alpha)) * (x - (beta / torch.sqrt(1.0 - ab)) * eps_pred)
                noise = torch.randn_like(x) if i > 0 else 0.0
                x = mean + torch.sqrt(beta) * noise
            
            # Save Image
            x_img = (x.clamp(-1, 1) + 1) / 2
            # Filename includes 'EMA' or 'LIVE'
            out_path = os.path.join(self.exp_dir, "samples", f"{prefix}_step{step:06d}.png")
            save_image(make_grid(x_img, nrow=nrow), out_path)
            return out_path

        # ema models sampls
        self.ema.copy_to(self.model) # EMA weights
        generate_and_save(prefix="EMA_grid", use_guidance=guidance_scale)

        # original models samples 
        self.model.load_state_dict(live_state_dict) # live weights
        generate_and_save(prefix="LIVE_grid", use_guidance=guidance_scale)
        
        self.model.train()

    def train(self, log_every=50):
        print(f"[{self.exp_name}] Poisson sampling q={self.q:.6f} | steps={self.steps} | ~{self.steps_per_epoch} steps/epoch")
        step = 0
        loader_iter = iter(self.train_loader)

        pbar = tqdm(range(1, self.steps + 1))
        for _ in pbar:
            step += 1
            epoch = (step - 1) // self.steps_per_epoch + 1
            is_epoch_end = (step % self.steps_per_epoch == 0)

            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(self.train_loader)
                batch = next(loader_iter)

            x0, y0 = batch
            x0 = x0.to(device, non_blocking=True)
            y0 = y0.to(device, non_blocking=True)
            B = x0.size(0)

            # enforce public input bound
            x0 = l2_clip_per_sample(x0, clip=self.dp_engine.X0)

            # classifier-free dropout on labels
            drop_mask = (torch.rand(B, device=device) < 0.1)
            y_train = y0.clone()
            y_train[drop_mask] = 10

            self.opt.zero_grad(set_to_none=True)

            loss_accum = 0.0
            for _k in range(self.K):
                t = torch.randint(0, self.cfg.T, (B,), device=device).long()
                eps = torch.randn_like(x0)
                ab = self.alpha_bars[t].view(-1, 1, 1, 1)
                x_t = torch.sqrt(ab) * x0 + torch.sqrt(1.0 - ab) * eps

                v_target = compute_v_target(x0, eps, t, self.alpha_bars)
                v_pred   = self.model(x_t, t, y_train)

                # bounded loss gradient (DP-valid)
                # loss = clipped_residual_mse(v_pred, v_target, R=self.dp_engine.R)
                loss = huber_loss(v_pred, v_target, delta=self.dp_engine.R)
                (loss / self.K).backward()
                loss_accum += float(loss.item())

            # DP step (adds noise + weight clipping periodically inside engine)
            self.dp_engine.add_noise_and_step(B)
            self.ema.update(self.model)

            # accountant (subsampled Gaussian mechanism, one step)
            if self.noise_multiplier > 0:
                self.accountant.step(noise_multiplier=self.noise_multiplier, sample_rate=self.q)

            avg_loss = loss_accum / self.K
            eps_now = self.get_epsilon()
            EPS_MAX = 8.0

            if eps_now >= 5.0: # capture Epsilon=5
                ckpt_dir = os.path.join(self.exp_dir, "checkpoints")
                os.makedirs(ckpt_dir, exist_ok=True)
                torch.save(self.model.state_dict(), os.path.join(ckpt_dir, "model_eps5.pth"))
                torch.save(self.ema.shadow, os.path.join(ckpt_dir, "ema_eps5.pth"))
                self.eps5_saved = True

            if eps_now >= 8.0: # capture Epsilon=8
                ckpt_dir = os.path.join(self.exp_dir, "checkpoints")
                os.makedirs(ckpt_dir, exist_ok=True)
                torch.save(self.model.state_dict(), os.path.join(ckpt_dir, "model_eps8.pth"))
                torch.save(self.ema.shadow, os.path.join(ckpt_dir, "ema_eps8.pth"))
                self.eps8_saved = True
                
            if eps_now >= EPS_MAX and not self.eps10_saved: 
                ckpt_dir = os.path.join(self.exp_dir, "checkpoints")
                os.makedirs(ckpt_dir, exist_ok=True)
                torch.save(self.model.state_dict(), os.path.join(ckpt_dir, "model_eps10.pth"))
                torch.save(self.ema.shadow, os.path.join(ckpt_dir, "ema_eps10.pth"))
                self.eps10_saved = True
    
            if eps_now >= EPS_MAX:
                print(f"Stopping: epsilon reached {eps_now:.2f} (>= {EPS_MAX}) at step {step}")
                self.save_checkpoint(step)
                self.sample_grid(step, guidance_scale=0.0)
                break

            if step % log_every == 0:
                pbar.set_postfix(
                    loss=f"{avg_loss:.4f}",
                    eps=f"{eps_now:.2f}",
                    B=B,
                    dG=f"{self.dp_engine.delta_global:.3e}",
                    ep=epoch
                )

            if step % log_every == 0 or step == 1:
                # history csv
                self.history.append({
                    "step": step,
                    "epoch": epoch,
                    "loss": avg_loss,
                    "epsilon": eps_now,
                    "batch_size_effective": B,
                    "delta_global": float(self.dp_engine.delta_global),
                })
                pd.DataFrame(self.history).to_csv(os.path.join(self.exp_dir, "metrics", "train_history.csv"), index=False)

                # diagnostics JSON (sensitivities + public bounds)
                diag = {
                    "step": step,
                    "epoch": epoch,
                    "loss": avg_loss,
                    "epsilon": eps_now,
                    "B_effective": int(B),
                    "delta_global": float(self.dp_engine.delta_global),
                    "delta_per_layer": self.dp_engine.delta_param,
                    "X_bounds": self.dp_engine.Xbounds,
                    "sigmas": self.dp_engine.sigma,
                    "X0_public": float(self.dp_engine.X0),
                    "Huber_R": float(self.dp_engine.R),
                    "noise_multiplier": float(self.noise_multiplier),
                    "C_weight": float(self.dp_engine.C),
                    "clip_every": int(self.dp_engine.clip_every),
                    "pi_iters": int(self.dp_engine.pi_iters),
                }
                save_json(diag, os.path.join(self.exp_dir, "diagnostics", f"diag_step{step:06d}.json"))

            # checkpoints & samples per Poisson-epoch
            if is_epoch_end or step == self.steps:
                # self.save_checkpoint(step)
                self.sample_grid(step, guidance_scale=0.0)


### FID/IS + downstream evaluation 

In [ ]:
def prepare_real_images_folder(dataset, output_folder, num_images=20000):
    os.makedirs(output_folder, exist_ok=True)
    existing = len([f for f in os.listdir(output_folder) if f.endswith(".png")])
    if existing >= num_images:
        return output_folder

    loader = DataLoader(dataset, batch_size=1, shuffle=True, drop_last=False)
    i = 0
    for img, _ in tqdm(loader, total=num_images, desc="Saving real images"):
        if i >= num_images:
            break
        img = img.squeeze(0)
        img01 = (img + 1) / 2
        save_image(img01, os.path.join(output_folder, f"real_{i:06d}.png"))
        i += 1
    return output_folder

In [ ]:
def compute_fid_and_is(gen_dir, real_dir, use_cuda=True):
    metrics = calculate_metrics(
        input1=gen_dir,
        input2=real_dir,
        cuda=use_cuda,
        fid=True,
        isc=True,
        verbose=False
    )
    return {
        "FID": float(metrics["frechet_inception_distance"]),
        "IS_mean": float(metrics["inception_score_mean"]),
        "IS_std": float(metrics["inception_score_std"]),
    }

In [ ]:
class SynthMNISTDataset(torch.utils.data.Dataset):
    def __init__(self, pt_path):
        d = torch.load(pt_path, map_location="cpu")
        self.images_u8 = d["images_u8"]
        self.labels = d["labels"]

    def __len__(self):
        return int(self.labels.shape[0])

    def __getitem__(self, idx):
        x = self.images_u8[idx].float() / 255.0
        x = (x * 2) - 1
        y = self.labels[idx].long()
        return x, y

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 10)
        )
    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Linear(64*7*7, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        h = self.conv(x)
        h = h.view(h.size(0), -1)
        return self.fc(h)

In [18]:
@torch.no_grad()
def eval_accuracy(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total

def train_classifier(model, train_loader, test_loader, device, epochs=10, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ce = nn.CrossEntropyLoss()

    for ep in range(1, epochs + 1):
        model.train()
        for x, y in tqdm(train_loader, desc=f"[clf] ep {ep}/{epochs}", leave=False):
            x = x.to(device); y = y.to(device)
            loss = ce(model(x), y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    return eval_accuracy(model, test_loader, device)

def downstream_eval_nn(synth_pt_path, mnist_test_ds, out_dir, device, batch_size=256):
    os.makedirs(out_dir, exist_ok=True)

    synth_ds = SynthMNISTDataset(synth_pt_path)
    synth_loader = DataLoader(synth_ds, batch_size=batch_size, shuffle=True, drop_last=False, num_workers=2)
    test_loader = DataLoader(mnist_test_ds, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=2)

    results = {}
    results["mlp"] = float(train_classifier(MLPClassifier(), synth_loader, test_loader, device=device, epochs=10, lr=1e-3))
    results["cnn"] = float(train_classifier(SimpleCNN(), synth_loader, test_loader, device=device, epochs=10, lr=1e-3))

    save_json(results, os.path.join(out_dir, "downstream_accuracy_nn.json"))
    pd.DataFrame([results]).to_csv(os.path.join(out_dir, "downstream_accuracy_nn.csv"), index=False)
    return results

In [ ]:
def load_synth_pt_to_numpy(synth_pt_path):
    d = torch.load(synth_pt_path, map_location="cpu")
    # Convert from uint8 [0, 255] to float [0, 1]
    X = d["images_u8"].numpy().astype(np.float32) / 255.0
    # Flatten (N, 1, 28, 28) -> (N, 784)
    X = X.reshape(X.shape[0], -1)
    y = d["labels"].numpy().astype(np.int64)
    return X, y

def load_real_mnist_test_to_numpy(mnist_test_ds):
    loader = DataLoader(mnist_test_ds, batch_size=512, shuffle=False)
    X_list, y_list = [], []
    for x, y in loader:
        # check if x is already [0, 1] or [-1, 1], map to [0, 1] to match synthetic loading
        x = ((x + 1) / 2).numpy().astype(np.float32)
        X_list.append(x.reshape(x.shape[0], -1))
        y_list.append(y.numpy().astype(np.int64))
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)

In [ ]:
# sklearn models as downstream classifiers: LogReg, DT, RF
def downstream_eval_sklearn(synth_pt_path, mnist_test_ds, out_dir, random_state=42):
    os.makedirs(out_dir, exist_ok=True)

    print("Loading data for sklearn eval...")
    X_train, y_train = load_synth_pt_to_numpy(synth_pt_path)
    X_test, y_test = load_real_mnist_test_to_numpy(mnist_test_ds)

    results = {}

    print("\n[sklearn] LogisticRegression...")
    logreg = LogisticRegression(max_iter=1000, solver="lbfgs", n_jobs=-1)
    logreg.fit(X_train, y_train)
    results["logreg"] = float(accuracy_score(y_test, logreg.predict(X_test)))
    print("  test_acc:", results["logreg"])

    print("\n[sklearn] DecisionTree...")
    dt = DecisionTreeClassifier(random_state=random_state)
    dt.fit(X_train, y_train)
    results["decision_tree"] = float(accuracy_score(y_test, dt.predict(X_test)))
    print("  test_acc:", results["decision_tree"])

    print("\n[sklearn] RandomForest...")
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=random_state)
    rf.fit(X_train, y_train)
    results["random_forest"] = float(accuracy_score(y_test, rf.predict(X_test)))
    print("  test_acc:", results["random_forest"])

    save_json(results, os.path.join(out_dir, "downstream_accuracy_sklearn.json"))
    pd.DataFrame([results]).to_csv(os.path.join(out_dir, "downstream_accuracy_sklearn.csv"), index=False)
    return results

In [ ]:
@torch.no_grad()
def export_png_folder_for_fid_is(model, ema: EMA, cfg, alphas, alpha_bars, betas,
                                 out_dir, n_samples=20000, batch_size=256, 
                                 guidance_scale=1.0, use_ema=True):
    os.makedirs(out_dir, exist_ok=True)
    model.eval()
    
    # Backup weights
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    
    # conditionally load EMA
    if use_ema:
        print("Loading EMA weights for PNG export")
        ema.copy_to(model)
    else:
        print("Using LIVE weights for PNG export")

    n_done = 0
    num_batches = (n_samples + batch_size - 1) // batch_size
    for _ in tqdm(range(num_batches), desc=f"Export PNGs (EMA={use_ema})"):
        cur = min(batch_size, n_samples - n_done)
        if cur <= 0:
            break
        x = torch.randn(cur, 1, 28, 28, device=device)
        y = torch.randint(0, 10, (cur,), device=device, dtype=torch.long)
        y_null = torch.full_like(y, 10)

        for i in reversed(range(cfg.T)):
            t = torch.full((cur,), i, device=device, dtype=torch.long)
            if guidance_scale > 0.0:
                v_cond = model(x, t, y)
                v_uncond = model(x, t, y_null)
                v_pred = v_uncond + guidance_scale * (v_cond - v_uncond)
            else:
                v_pred = model(x, t, y)

            eps_pred, _ = v_to_eps_and_x0(x, t, v_pred, alpha_bars)
            alpha = alphas[i]
            ab = alpha_bars[i]
            beta  = betas[i]
            mean = (1.0 / torch.sqrt(alpha)) * (x - (beta / torch.sqrt(1.0 - ab)) * eps_pred)
            noise = torch.randn_like(x) if i > 0 else 0.0
            x = mean + torch.sqrt(beta) * noise

        x01 = (x.clamp(-1, 1) + 1) / 2
        for k in range(cur):
            save_image(x01[k].cpu(), os.path.join(out_dir, f"gen_{n_done+k:06d}.png"))
        n_done += cur

    # restore
    model.load_state_dict(backup)
    model.train()
    return out_dir

In [ ]:
@torch.no_grad()
def generate_synthetic_pt(model, ema: EMA, cfg, alphas, alpha_bars, betas,
                          out_path, n_samples=60000, batch_size=256, 
                          guidance_scale=1.0, use_ema=True): 
    model.eval()
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    
    if use_ema:
        print("Loading EMA weights for PT generation")
        ema.copy_to(model)
    else:
        print("Using LIVE weights for PT generation")

    img_list, lab_list = [], []
    n_done = 0
    num_batches = (n_samples + batch_size - 1) // batch_size

    for _ in tqdm(range(num_batches), desc=f"Generate synth {n_samples} (EMA={use_ema})"):
        cur = min(batch_size, n_samples - n_done)
        if cur <= 0:
            break

        x = torch.randn(cur, 1, 28, 28, device=device)
        y = torch.randint(0, 10, (cur,), device=device, dtype=torch.long)
        y_null = torch.full_like(y, 10)

        for i in reversed(range(cfg.T)):
            t = torch.full((cur,), i, device=device, dtype=torch.long)
            if guidance_scale > 0.0:
                v_cond = model(x, t, y)
                v_uncond = model(x, t, y_null)
                v_pred = v_uncond + guidance_scale * (v_cond - v_uncond)
            else:
                v_pred = model(x, t, y)

            eps_pred, _ = v_to_eps_and_x0(x, t, v_pred, alpha_bars)
            alpha = alphas[i]
            ab = alpha_bars[i]
            beta = betas[i]
            mean = (1.0 / torch.sqrt(alpha)) * (x - (beta / torch.sqrt(1.0 - ab)) * eps_pred)
            x = mean + (torch.sqrt(beta) * torch.randn_like(x) if i > 0 else 0.0)

        x01 = (x.clamp(-1, 1) + 1) / 2
        x_u8 = (x01 * 255.0).round().to(torch.uint8).cpu()

        img_list.append(x_u8)
        lab_list.append(y.cpu())
        n_done += cur

    images = torch.cat(img_list, dim=0)[:n_samples]
    labels = torch.cat(lab_list, dim=0)[:n_samples]

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    torch.save({"images_u8": images, "labels": labels}, out_path)

    model.load_state_dict(backup)
    model.train()
    return out_path

In [ ]:
def evaluate_trainer(trainer: DiffusionDPTrainer,
                     mnist_train_ds,
                     mnist_test_ds,
                     fid_is_n=20000,
                     synth_n=60000,
                     guidance_scale=1.0,
                     use_ema=True,
                     eps_tag="latest"):
    
    model_type = "ema" if use_ema else "live"
    
    metrics_dir = os.path.join(trainer.exp_dir, "metrics")
    os.makedirs(metrics_dir, exist_ok=True)
    synth_pt_path = os.path.join(trainer.exp_dir, "synthetic_data", f"synth_{synth_n}_w{guidance_scale}_{model_type}_{eps_tag}.pt")
    
    synth_pt = generate_synthetic_pt(
        trainer.model, trainer.ema, trainer.cfg,
        trainer.alphas, trainer.alpha_bars, trainer.betas,
        out_path=synth_pt_path,
        n_samples=synth_n,
        batch_size=256,
        guidance_scale=guidance_scale,
        use_ema=use_ema 
    )

    real_dir = os.path.join(RUN_DIR, "fid_is_real_mnist_train_20k")
    prepare_real_images_folder(mnist_train_ds, real_dir, num_images=fid_is_n)

    gen_dir = os.path.join(trainer.exp_dir, "fid_is", f"gen_{fid_is_n}_w{guidance_scale}_{model_type}_{eps_tag}")
    
    export_png_folder_for_fid_is(
        trainer.model, trainer.ema, trainer.cfg,
        trainer.alphas, trainer.alpha_bars, trainer.betas,
        out_dir=gen_dir,
        n_samples=fid_is_n,
        batch_size=256,
        guidance_scale=guidance_scale,
        use_ema=use_ema
    )

    fidis = compute_fid_and_is(gen_dir, real_dir, use_cuda=torch.cuda.is_available())
    save_json(fidis, os.path.join(metrics_dir, f"fid_is_{fid_is_n}_{model_type}_w{guidance_scale}_{eps_tag}.json"))

    print(f"\n--- Running Downstream Evaluation (Model: {model_type}, w={guidance_scale}, {eps_tag}) ---")
    nn_res = downstream_eval_nn(synth_pt, mnist_test_ds, metrics_dir, device=device)
    sklearn_res = downstream_eval_sklearn(synth_pt, mnist_test_ds, metrics_dir)

    combined = {
        "exp_name": trainer.exp_name,
        "epsilon_final": trainer.get_epsilon(),
        "tag": eps_tag, 
        "model_type": model_type,
        "guidance_scale": guidance_scale,
        "FID": fidis["FID"],
        "IS_mean": fidis["IS_mean"],
        "IS_std": fidis["IS_std"],
        "downstream_nn": nn_res,
        "downstream_sklearn": sklearn_res, 
    }
    
    save_json(combined, os.path.join(metrics_dir, f"all_metrics_{model_type}_w{guidance_scale}_{eps_tag}.json"))
    return combined

In [19]:
ddpm_cfg = DDPMConfig(timesteps=1000, beta_start=1e-4, beta_end=0.02)

N = len(fmnist_train)
q= 256 / N  # same for fashion mnist and mnist as their len is same (60000)

X0_PUBLIC = 28.0 
Huber_R = 1.0
lr = 3e-4
K_MULT = 32
EMA_DECAY = 0.95

# DP settings
C_WEIGHT_PUBLIC = 1.1
C_WEIGHT = 1.2
NOISE_MULT = 1.2 # fror private training only
DELTA = 1e-5

PI_ITERS = 2
CLIP_EVERY = 10

STEPS_PUBLIC = 8000
STEPS_PRIVATE = 10000

In [20]:
print("\n\n" + "="*50)
print("PHASE 1: PUBLIC PRE-TRAINING (Fashion-MNIST)")
print("="*50)
    
phase1_model = CondDiffUNet(n_classes=11, use_lip_norm=False, base_channels=64).to(device)

phase1_trainer = DiffusionDPTrainer(
    exp_name="Lip_DP_diff_Phase1_Public_FashionMNIST",
    model=phase1_model,
    ddpm_cfg=ddpm_cfg,
    train_dataset=fmnist_train, 
    test_dataset=fmnist_test,
    q_sample_rate=q, 
    steps=STEPS_PUBLIC,                 
    lr=1e-4,
    K_noise_mult=16,
    ema_decay=EMA_DECAY,
    C_weight=C_WEIGHT_PUBLIC,
    noise_multiplier=0.0,       #no noise for public training
    delta=DELTA,
    X0_public=X0_PUBLIC,
    Huber_R=Huber_R,
    pi_iters=PI_ITERS,
    clip_every=CLIP_EVERY,
)



PHASE 1: PUBLIC PRE-TRAINING (Fashion-MNIST)


In [79]:
phase1_dir = phase1_trainer.train()

[Lip_DP_diff_Phase1_Public_FashionMNIST] Poisson sampling q=0.004267 | steps=8000 | ~234 steps/epoch


  0%|                                            | 1/8000 [00:00<1:13:00,  1.83it/s]

[DP step 1] B=280 | Δ_global=6.130e-03 | avg_grad_RMS=7.795e-03 | Approx SNR=inf


  0%|                                            | 2/8000 [00:00<1:05:00,  2.05it/s]

[DP step 2] B=245 | Δ_global=7.005e-03 | avg_grad_RMS=1.299e-02 | Approx SNR=inf


...

 65%|▋| 5200/8000 [43:59<21:32,  2.17it/s, B=269, dG=6.961e-03, ep=23, eps=0.00, los

[DP step 5200] B=269 | Δ_global=6.961e-03 | avg_grad_RMS=3.424e-04 | Approx SNR=inf


 66%|▋| 5250/8000 [44:22<21:23,  2.14it/s, B=269, dG=6.943e-03, ep=23, eps=0.00, los

[DP step 5250] B=269 | Δ_global=6.943e-03 | avg_grad_RMS=8.965e-04 | Approx SNR=inf


 66%|▋| 5280/8000 [44:36<22:16,  2.03it/s, B=269, dG=6.943e-03, ep=23, eps=0.00, losIOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [21]:
phase1_dir = phase1_trainer.exp_dir
phase1_ckpt = os.path.join(phase1_dir, "checkpoints", "model_latest.pth")
print(f"Phase 1 Complete. Checkpoint saved: {phase1_ckpt}")

Phase 1 Complete. Checkpoint saved: ./diffusion_experiments/14.02.2026/Lip_DP_diff_Phase1_Public_FashionMNIST/checkpoints/model_latest.pth


In [22]:
q= 4096 / N

In [23]:
print("\n\n" + "="*50)
print("PHASE 2: PRIVATE FINE-TUNING (MNIST)")
print("="*50)
    
# init fresh model
phase2_model = CondDiffUNet(n_classes=11, use_lip_norm=False, base_channels = 64).to(device)

# load weights from Phase 1 public training
print(f"Loading weights from Phase 1: {phase1_ckpt}")
phase2_model.load_state_dict(torch.load(phase1_ckpt))



PHASE 2: PRIVATE FINE-TUNING (MNIST)
Loading weights from Phase 1: ./diffusion_experiments/14.02.2026/Lip_DP_diff_Phase1_Public_FashionMNIST/checkpoints/model_latest.pth


/tmp/ipykernel_372683/1833304338.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  phase2_model.load_state_dict(torch.load(phase1_ckpt))


<All keys matched successfully>

In [26]:
#  Private fine-tuning
phase2_trainer = DiffusionDPTrainer(
    exp_name="Lip_DP_diff_Phase2_Private_FineTuning_MNIST",
    model=phase2_model,
    ddpm_cfg=ddpm_cfg,
    train_dataset=mnist_train,  # MNIST
    test_dataset=mnist_test,
    steps=STEPS_PRIVATE,                 
    q_sample_rate=q,
    lr=1e-4,                    
    K_noise_mult=16,
    ema_decay=EMA_DECAY,
    C_weight=C_WEIGHT,
    noise_multiplier=1.2,       
    delta=1e-5,
    X0_public=X0_PUBLIC,
    Huber_R=Huber_R,
    pi_iters=PI_ITERS,
    clip_every=CLIP_EVERY
)

In [27]:
start_time = time.time()

phase2_trainer.train()

end_time = time.time()
elapsed = end_time - start_time

print(f"Phase 2 Complete. Final Epsilon: {phase2_trainer.get_epsilon():.2f}")
print(f"Training time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

[Lip_DP_diff_Phase2_Private_FineTuning_MNIST] Poisson sampling q=0.068267 | steps=10000 | ~15 steps/epoch


  0%|                                          | 1/10000 [00:12<35:57:36, 12.95s/it]

[DP step 1] B=4028 | Δ_global=5.350e-04 | avg_grad_RMS=3.559e-04 | Approx SNR=5.543e-01


  0%|                                          | 2/10000 [00:27<38:29:23, 13.86s/it]

[DP step 2] B=4114 | Δ_global=5.238e-04 | avg_grad_RMS=5.666e-03 | Approx SNR=9.014e+00


...

[DP step 500] B=4060 | Δ_global=5.816e-04 | avg_grad_RMS=1.990e-04 | Approx SNR=2.851e-01


  5%| | 510/10000 [1:28:48<25:37:36,  9.72s/it, B=4060, dG=5.816e-04, ep=34, eps=8.4

Stopping: epsilon reached 8.50 (>= 8.5) at step 511


  5%| | 510/10000 [1:29:05<27:37:54, 10.48s/it, B=4060, dG=5.816e-04, ep=34, eps=8.4

Phase 2 Complete. Final Epsilon: 8.50
Training time: 5345.83 seconds (89.10 minutes)


In [30]:
print("\n>>> Evaluating EMA Model (w=2.0)...")
metrics_ema_w2 = evaluate_trainer(
    phase2_trainer, mnist_train, mnist_test, fid_is_n=20000, synth_n=60000, 
    guidance_scale=2.0, use_ema=True)


>>> Evaluating EMA Model (w=2.0)...
--> Loading EMA weights for PT generation...


Saving real images: 100%|███████████████████| 20000/20000 [00:09<00:00, 2014.03it/s]


--> Loading EMA weights for PNG export...


Export PNGs (EMA=True): 100%|███████████████████████| 79/79 [22:48<00:00, 17.33s/it]
/opt/conda/lib/python3.11/site-packages/torch_fidelity/datasets.py:16: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes())).view(height, width, 3)
/tmp/ipykernel_372683/2262210266.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `T


--- Running Downstream Evaluation (Model: ema, w=2.0, latest) ---


Loading data for sklearn eval...


/tmp/ipykernel_372683/3928588880.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(synth_pt_path, map_location="cpu")



[sklearn] LogisticRegression...


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  test_acc: 0.8935

[sklearn] DecisionTree...
  test_acc: 0.8161

[sklearn] RandomForest...
  test_acc: 0.9225


In [31]:
print("\n>>> Evaluating EMA Model (w=0.0)...")
metrics_ema_w0 = evaluate_trainer(
    phase2_trainer, mnist_train, mnist_test, fid_is_n=20000, synth_n=60000, 
    guidance_scale=0.0, use_ema=True)


>>> Evaluating EMA Model (w=0.0)...
--> Loading EMA weights for PT generation...


Generate synth 60000 (EMA=True): 100%|████████████| 235/235 [34:01<00:00,  8.69s/it]


--> Loading EMA weights for PNG export...


Export PNGs (EMA=True): 100%|███████████████████████| 79/79 [11:30<00:00,  8.74s/it]
/tmp/ipykernel_372683/2262210266.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d 


--- Running Downstream Evaluation (Model: ema, w=0.0, latest) ---


Loading data for sklearn eval...


/tmp/ipykernel_372683/3928588880.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(synth_pt_path, map_location="cpu")



[sklearn] LogisticRegression...


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  test_acc: 0.9094

[sklearn] DecisionTree...
  test_acc: 0.827

[sklearn] RandomForest...
  test_acc: 0.9459


In [32]:
print("\n>>> Evaluating LIVE Model (w=2.0)...")
metrics_live_w2 = evaluate_trainer(
    phase2_trainer, mnist_train, mnist_test, fid_is_n=20000, synth_n=60000, 
    guidance_scale=2.0, use_ema=False)


>>> Evaluating LIVE Model (w=2.0)...
--> Using LIVE weights for PT generation...


Generate synth 60000 (EMA=False): 100%|█████████| 235/235 [1:07:45<00:00, 17.30s/it]


--> Using LIVE weights for PNG export...


Export PNGs (EMA=False): 100%|██████████████████████| 79/79 [22:47<00:00, 17.31s/it]
/tmp/ipykernel_372683/2262210266.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d 


--- Running Downstream Evaluation (Model: live, w=2.0, latest) ---


Loading data for sklearn eval...


/tmp/ipykernel_372683/3928588880.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(synth_pt_path, map_location="cpu")



[sklearn] LogisticRegression...


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  test_acc: 0.9006

[sklearn] DecisionTree...
  test_acc: 0.8111

[sklearn] RandomForest...
  test_acc: 0.9258


In [33]:
print("\n>>> Evaluating LIVE Model (w=0.0)...")
metrics_live_w0 = evaluate_trainer(
    phase2_trainer, mnist_train, mnist_test, fid_is_n=20000, synth_n=60000, 
    guidance_scale=0.0, use_ema=False)


>>> Evaluating LIVE Model (w=0.0)...
--> Using LIVE weights for PT generation...


Generate synth 60000 (EMA=False): 100%|███████████| 235/235 [34:02<00:00,  8.69s/it]


--> Using LIVE weights for PNG export...


Export PNGs (EMA=False): 100%|██████████████████████| 79/79 [11:30<00:00,  8.74s/it]
/tmp/ipykernel_372683/2262210266.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d 


--- Running Downstream Evaluation (Model: live, w=0.0, latest) ---


Loading data for sklearn eval...


/tmp/ipykernel_372683/3928588880.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(synth_pt_path, map_location="cpu")



[sklearn] LogisticRegression...


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  test_acc: 0.9101

[sklearn] DecisionTree...
  test_acc: 0.8293

[sklearn] RandomForest...
  test_acc: 0.9483
